# 02 â€” Room occupancy under a sensor outage
**Assessed group practical â€¢ three students â€¢ English**

Use the worked sprinkler notebook as a recipe. Complete the **four short modelling
cells** below; supplied helper functions handle data loading and scoring.
Do not write a new inference engine, data scraper or preprocessing pipeline.

The task is to compare the same fitted model on the same test records with full
sensor evidence and with one assigned sensor feature omitted. Explain the result,
including a small or unexpected effect. There is no performance leaderboard.

**Submission:** this notebook, saved with outputs, and three presentation slides in
PDF. Common deadline: end of lecture 2. No separate report or
individual essay. The five-minute oral will involve questions for each member of the group.

This is a starter: cells tagged `student-task` deliberately raise
`NotImplementedError` until completed. Real prepared data are supplied by the
lecturer before class. The notebook does not download data or use a synthetic
fallback for the assessment.

## 0. Dependencies and auxiliary functions

In [1]:
%pip install pandas pgmpy

/Users/giovani/Developer/University/IMT Mines Alès/Advanced Machine Learning/Advanced-Machine-Learning/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Utility functions, run this cell first. Do not edit it. The code is provided for your convenience and to ensure consistency across student submissions.

"""Supplied infrastructure for the occupancy practical; modelling stays in the notebook."""
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from collections.abc import Callable, Mapping
import hashlib
import json
import numpy as np
import pandas as pd

FEATURES = ['T', 'L', 'S', 'C', 'M']
VARIABLES = ['O', *FEATURES]
THRESHOLD = 1.0 / 6.0


@dataclass(frozen=True)
class CourseData:
    train: pd.DataFrame
    test: pd.DataFrame
    state_names: dict[str, list[int]]
    manifest: dict


def locate_data(start: Path | None = None) -> Path:
    """Find the distributed data folder from a notebook or the package root."""
    here = (start or Path.cwd()).resolve()
    for folder in [here / 'data', here.parent / 'data', here]:
        if (folder / 'manifest.json').is_file():
            return folder
    raise FileNotFoundError(
        'Prepared occupancy data are missing. Ask the lecturer for the data folder. '
        'Before class, the lecturer runs: python tools/prepare_occupancy.py '
        '--input Occupancy_Estimation.csv --output-dir data. '
        'The sprinkler example does not require these data.'
    )


def load_course_data(folder: Path | str, *, allow_synthetic: bool = False) -> CourseData:
    folder = Path(folder)
    meta = json.loads((folder / 'manifest.json').read_text(encoding='utf-8'))
    if meta['source_kind'] != 'uci864' and not allow_synthetic:
        raise ValueError(
            'This is a synthetic test fixture, not the assessed UCI dataset.')
    frames = []
    for split in ['train', 'test']:
        p = folder / f'occupancy_{split}.csv'
        actual = hashlib.sha256(p.read_bytes()).hexdigest()
        if actual != meta['files'][p.name]['sha256']:
            raise ValueError(f'{p.name} differs from the published manifest.')
        frame = pd.read_csv(p)
        required = ['record_id', 'timestamp', *VARIABLES]
        if list(frame.columns) != required:
            raise ValueError(
                f'Unexpected columns in {p.name}. Expected {required}.')
        if frame[required].isna().any().any():
            raise ValueError(f'Missing training/test values in {p.name}.')
        for var in VARIABLES:
            legal = meta['state_names'][var]
            if not frame[var].isin(legal).all():
                raise ValueError(f'Unknown state of {var} in {p.name}.')
            frame[var] = frame[var].astype(int)
        if len(frame) != meta['splits'][split]['rows']:
            raise ValueError(f'Wrong row count in {p.name}.')
        if not pd.to_datetime(frame.timestamp).is_monotonic_increasing:
            raise ValueError(f'{p.name} is not chronologically sorted.')
        frames.append(frame)
    train, test = frames
    if set(train.record_id) & set(test.record_id):
        raise ValueError('Training and test records overlap.')
    if pd.to_datetime(train.timestamp).max() >= pd.to_datetime(test.timestamp).min():
        raise ValueError(
            'The test period must follow the training period strictly.')
    return CourseData(train, test, meta['state_names'], meta)


def outage_for_group(group: str) -> str:
    """Assignment fixed before any student result is seen."""
    group = group.strip().upper()
    if not (len(group) == 3 and group[0] == 'G' and group[1:].isdigit()):
        raise ValueError('Use a group identifier from G01 to G19.')
    number = int(group[1:])
    if not 1 <= number <= 19:
        raise ValueError('Use a group identifier from G01 to G19.')
    return 'C' if number <= 4 else 'M' if number <= 8 else 'L' if number <= 12 else 'S' if number <= 16 else 'T'


def predict_cached(rows: pd.DataFrame,
                   make_evidence: Callable[[pd.Series, str | None], Mapping[str, int]],
                   probability: Callable[[Mapping[str, int]], float],
                   omitted: str | None = None) -> np.ndarray:
    """Call the student's query once per unique evidence pattern, preserving row order.

    The helper checks that the evidence is exactly the declared sensor subset;
    it never reads O as evidence and never learns model parameters.
    """
    if omitted is not None and omitted not in FEATURES:
        raise ValueError('The omitted variable must be a sensor feature.')
    expected_keys = set(FEATURES) - ({omitted} if omitted else set())
    cache: dict[tuple, float] = {}
    result = []
    # Give student code only sensor columns: targets and metadata are unavailable here.
    for _, row in rows[FEATURES].iterrows():
        evidence = dict(make_evidence(row, omitted))
        if set(evidence) != expected_keys:
            raise ValueError(
                f'Evidence keys must be {sorted(expected_keys)}; got {sorted(evidence)}.')
        key = tuple(sorted((k, int(v)) for k, v in evidence.items()))
        if key not in cache:
            p = float(probability(dict(key)))
            if not np.isfinite(p) or not 0 <= p <= 1:
                raise ValueError(
                    'An occupancy posterior is nonfinite or outside [0,1].')
            cache[key] = p
        result.append(cache[key])
    return np.asarray(result, dtype=float)


def binary_metrics(y: np.ndarray | pd.Series, p: np.ndarray,
                   threshold: float = THRESHOLD) -> dict[str, float | int]:
    y = np.asarray(y)
    p = np.asarray(p, dtype=float)
    if y.ndim != 1 or p.shape != y.shape or len(y) == 0:
        raise ValueError(
            'Labels and probabilities must be nonempty aligned vectors.')
    if not np.isin(y, [0, 1]).all() or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError(
            'Use binary labels and finite probabilities in [0,1].')
    if not 0 <= threshold <= 1:
        raise ValueError('Threshold must be in [0,1].')
    decisions = p >= threshold  # Occupied is the declared tie decision.
    tn = int(np.sum((y == 0) & ~decisions))
    fp = int(np.sum((y == 0) & decisions))
    fn = int(np.sum((y == 1) & ~decisions))
    tp = int(np.sum((y == 1) & decisions))
    return {'n': len(y), 'brier': float(np.mean((p-y)**2)),
            'mean_loss': float((fp+5*fn)/len(y)),
            'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}


def results_table(y: pd.Series, predictions: Mapping[str, np.ndarray]) -> pd.DataFrame:
    return pd.DataFrame({name: binary_metrics(y, p) for name, p in predictions.items()}).T.rename_axis('condition')

In [ ]:
from pathlib import Path
import sys
from importlib.metadata import version
import numpy as np
import pandas as pd
from IPython.display import display
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination


GROUP = "G12"   # Replace with your assigned identifier.
MEMBERS = ["Giovani Mambrim Leme", "Yanis Blot--EL Mazouzi", "Abdelbasset El Hamrit", "Dhruv Gupta"]
OMITTED = outage_for_group(GROUP)
print("Group:", GROUP, "outage:", OMITTED, "pgmpy:", version("pgmpy"))

Group: G12 outage: L pgmpy: 1.1.2


## 1. Read the supplied data contract
`O=1` means at least one occupant. Sensor features are temperature `T`, light `L`,
sound `S`, COâ‚‚ `C` and motion `M`. Continuous features were discretised using **only
the earlier training period**; use the recorded states, not new test-fitted bins.

`record_id` and `timestamp` identify the rows; neither is evidence.
The original occupancy count and the derived label `O` are never sensor evidence.

In [5]:
data = load_course_data(locate_data())
train, test, state_names = data.train, data.test, data.state_names
print("Protocol:", data.manifest["protocol"])
print("Source hash:", data.manifest["source_sha256"])
print("Split:", data.manifest["splits"])
print("State dictionary:", state_names)
display(train.head())

Protocol: imt-bn-two-mornings-v1
Source hash: c090ee5c94b61762bfec9d22767864490b51030f98cca030f02822468db25d9c
Split: {'train': {'rows': 7090, 'first_timestamp': '2017-12-22T10:49:41', 'last_timestamp': '2017-12-26T00:35:20', 'class_counts': {'0': 5483, '1': 1607}}, 'test': {'rows': 3039, 'first_timestamp': '2017-12-26T00:35:51', 'last_timestamp': '2018-01-11T09:00:09', 'class_counts': {'0': 2745, '1': 294}}}
State dictionary: {'O': [0, 1], 'M': [0, 1], 'T': [0, 1, 2], 'L': [0, 1], 'S': [0, 1, 2], 'C': [0, 1, 2]}


,record_id,timestamp,O,T,L,S,C,M
0,0,2017-12-22T10:49:41,1,0,1,2,1,0
1,1,2017-12-22T10:50:12,1,0,1,2,1,0
2,2,2017-12-22T10:50:42,1,0,1,2,1,0
3,3,2017-12-22T10:51:13,1,0,1,2,1,0
4,4,2017-12-22T10:51:44,1,0,1,2,1,0


## 2. Task A â€” Represent the fixed model
Use the factorisation
\[
P(O,T,L,S,C,M)=P(O)P(T\mid O)P(L\mid O)P(S\mid O)P(C\mid O)P(M\mid O).
\]
Create the five edges and the model. Use the same constructor as in sprinkler section 2.
This assumes conditional independence of sensor features given occupancy.

In [8]:
# Task A: define edges and model; retain all six VARIABLES.
# Example API pattern: DiscreteBayesianNetwork(list_of_parent_child_pairs).
EDGES = [
    ('O', 'T'),
    ('O', 'L'),
    ('O', 'S'),
    ('O', 'C'),
    ('O', 'M'),
]

model = DiscreteBayesianNetwork(EDGES)
model.add_nodes_from(VARIABLES)

print("Children of O:", sorted(model.get_children('O')))
print("Parents of T:", sorted(model.get_parents('T')))
print("Graph's edged:", model.edges())
print("Graph's nodes:", model.nodes())
print("O's Markov Blanket:", model.get_markov_blanket('O'))
print(model.get_independencies())

Children of O: ['C', 'L', 'M', 'S', 'T']
Parents of T: ['O']
Graph's edged: [('O', 'T'), ('O', 'L'), ('O', 'S'), ('O', 'C'), ('O', 'M')]
Graph's nodes: ['O', 'T', 'L', 'S', 'C', 'M']
O's Markov Blanket: ['T', 'L', 'C', 'S', 'M']
(M ⟂ S | O)
(L ⟂ T | O)
(C ⟂ S | O)
(L ⟂ M | O)
(L ⟂ S | O)
(C ⟂ L | O)
(S ⟂ T | O)
(C ⟂ M | O)
(C ⟂ T | O)
(M ⟂ T | O)


In [7]:
assert set(model.nodes()) == set(VARIABLES)
assert set(model.edges()) == {("O", feature) for feature in FEATURES}

## 3. Task B â€” Learn CPDs, rather than copying sprinkler probabilities
Transpose sprinkler section 8. Construct a `BayesianEstimator` from `model`,
`train[VARIABLES]` and `state_names`. Request Dirichlet posterior-mean estimates
with **one pseudocount per cell**, then attach the returned CPDs.
Use `n_jobs=1` for this small model.

In [9]:
# Task B: estimate CPDs using only train[VARIABLES] and attach them to model.
learned_model = DiscreteBayesianNetwork(EDGES)
learned_model.add_nodes_from(VARIABLES)
estimator = BayesianEstimator(learned_model, train, state_names=state_names)
learned_cpds = estimator.get_parameters(
    prior_type="dirichlet", pseudo_counts=1, n_jobs=1
)
learned_model.add_cpds(*learned_cpds)
assert learned_model.check_model()

/var/folders/bm/6508_k4d0nl3sbkvytgtw2vr0000gn/T/ipykernel_10021/3950558951.py:4: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  estimator = BayesianEstimator(learned_model, train, state_names=state_names)


In [17]:
assert learned_model.check_model()
for cpd in learned_model.get_cpds():
    np.testing.assert_allclose(cpd.get_values().sum(axis=0), 1.0, atol=1e-12, rtol=0)

# Supplied numerical check; explain the numerator and denominator orally.
child, child_state, parent_state = "M", 1, 1
subset = train.loc[train.O == parent_state, child]
print("Subset length:", len(subset))
print("Count of child_state in subset:", (subset == child_state).sum())
print("State names for child:", state_names[child])
manual = (int((subset == child_state).sum()) + 1) / (len(subset) + len(state_names[child]))
computed = float(learned_model.get_cpds(child).get_value(**{child: child_state, "O": parent_state}))
np.testing.assert_allclose(manual, computed, atol=1e-12, rtol=0)
print("P(M=1 | O=1), hand count and CPD:", manual, computed)

Subset length: 1607
Count of child_state in subset: 939
State names for child: [0, 1]
P(M=1 | O=1), hand count and CPD: 0.584213797389683 0.584213797389683


## 4. Task C â€” Query occupancy
Create a `VariableElimination` object. Complete the function that returns
$P(O=1\mid\mathrm{evidence})$. Adapt the query and named-state access in sprinkler
sections 5â€“6. Evidence is a dictionary of observed sensor states.

In [32]:
inference = VariableElimination(learned_model)

def occupancy_probability(evidence: dict[str, int]) -> float:
    """Return the state-1 posterior; O itself must not be in evidence."""
    if not set(evidence).issubset(FEATURES):
        raise ValueError("Evidence may contain sensor features only.")
    # Query O and return float(result.get_value(O=1)).
    road = inference.query(
        variables=['O'],
        evidence=evidence,
        elimination_order="MinFill", show_progress=False,
    )
    p = float(road.get_value(**{'O': 1}))
    if not np.isfinite(p):
        raise ValueError("Nonfinite posterior: check for impossible evidence.")

    return p

In [33]:
# The same smoothed root CPD defines the prior-only baseline and no-evidence check.
prior = float(learned_model.get_cpds("O").get_value(O=1))
np.testing.assert_allclose(occupancy_probability({}), prior, atol=1e-12, rtol=0)
print("Fitted occupancy prior:", prior)

Fitted occupancy prior: 0.22673434856175972


## 5. Task D â€” Form evidence; omit an unavailable sensor
Return all five sensor states for the full model. When `omitted` is a sensor name,
leave its key out. A missing motion sensor is **not** `M=0`.
The supplied predictor passes only sensor columns to this function and checks its keys.

In [35]:
def make_evidence(row: pd.Series, omitted: str | None = None) -> dict[str, int]:
    """Read FEATURES from this row, except the variable named in omitted."""
    if omitted is not None and omitted not in FEATURES:
        raise ValueError('The omitted variable must be a sensor feature.')
    return {feature: int(row[feature]) for feature in FEATURES if feature != omitted}


In [36]:
# All three conditions use exactly these test records, in this order.
predictions = {
    "prior only": np.full(len(test), prior),
    "full sensors": predict_cached(test, make_evidence, occupancy_probability),
    f"without {OMITTED}": predict_cached(test, make_evidence, occupancy_probability, omitted=OMITTED),
}
summary = results_table(test.O, predictions)
display(summary.round(5))

,n,brier,mean_loss,TN,FP,FN,TP
condition,,,,,,,
prior only,3039.0,0.10428,0.90326,0.0,2745.0,0.0,294.0
full sensors,3039.0,0.02350,0.08292,2743.0,2.0,50.0,244.0
without L,3039.0,0.01118,0.04673,2608.0,137.0,1.0,293.0


## 6. Interpret the experiment
The helper reports Brier score (mean squared probability error), confusion counts
and mean loss. The supplied hypothetical costs are false occupied = 1, false empty = 5,
correct decisions = 0. The fixed decision is occupied for $p\ge1/6$ (including ties).
These are educational costs, not measured building savings.

**Complete these short group notes:**

- Which comparison isolates the effect of omitting your assigned sensor?
- What changed, by how much, and what remained controlled?
- What does the result suggest, and what important claim does it not establish?

Do not tune the graph, bins, smoothing or threshold on the test outcomes.

In [43]:
# Calculates accuracy, precision and recall for each condition; the last three columns are the confusion matrix counts.
prior_accuracy = float(summary.loc["prior only", "TP"] + summary.loc["prior only", "TN"]) / summary.loc["prior only", "n"]
full_accuracy = float(summary.loc["full sensors", "TP"] + summary.loc["full sensors", "TN"]) / summary.loc["full sensors", "n"]
ommited_accuracy = float(summary.loc[f"without {OMITTED}", "TP"] + summary.loc[f"without {OMITTED}", "TN"]) / summary.loc[f"without {OMITTED}", "n"]

prior_precision = float(summary.loc["prior only", "TP"]) / (summary.loc["prior only", "TP"] + summary.loc["prior only", "FP"])
full_precision = float(summary.loc["full sensors", "TP"]) / (summary.loc["full sensors", "TP"] + summary.loc["full sensors", "FP"])
ommited_precision = float(summary.loc[f"without {OMITTED}", "TP"]) / (summary.loc[f"without {OMITTED}", "TP"] + summary.loc[f"without {OMITTED}", "FP"])

prior_recall = float(summary.loc["prior only", "TP"]) / (summary.loc["prior only", "TP"] + summary.loc["prior only", "FN"])
full_recall = float(summary.loc["full sensors", "TP"]) / (summary.loc["full sensors", "TP"] + summary.loc["full sensors", "FN"])
ommited_recall = float(summary.loc[f"without {OMITTED}", "TP"]) / (summary.loc[f"without {OMITTED}", "TP"] + summary.loc[f"without {OMITTED}", "FN"])

full_probabilities = predictions["full sensors"]

false_negative_mask = (
    (test["O"].to_numpy() == 1)
    & (full_probabilities < THRESHOLD)
)

false_negative_records = test.loc[false_negative_mask].copy()

counts_by_L = (
    false_negative_records["L"]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename(index={0: "L = 0", 1: "L = 1"})
)

print("Total false negatives:", len(false_negative_records))
print(counts_by_L)

# Compute the difference in all the metrics in summary (from brier to TP) between the prior, full sensors and the omitted sensor condition.
from itertools import combinations

metrics = ["n", "brier", "mean_loss", "TN", "FP", "FN", "TP"]

conditions = [
    "prior only",
    "full sensors",
    "without L",
]

differences = {}

for first, second in combinations(conditions, 2):
    differences[f"{first} - {second}"] = (
        summary.loc[first, metrics] - summary.loc[second, metrics]
    )

differences = pd.DataFrame(differences).T

print("Accuracies:")
print(f"Prior only: {prior_accuracy:.5f}")
print(f"Full sensors: {full_accuracy:.5f}")
print(f"Without {OMITTED}: {ommited_accuracy:.5f}")
print("\nPrecisions:")
print(f"Prior only: {prior_precision:.5f}")
print(f"Full sensors: {full_precision:.5f}")
print(f"Without {OMITTED}: {ommited_precision:.5f}")
print("\nRecalls:")
print(f"Prior only: {prior_recall:.5f}")
print(f"Full sensors: {full_recall:.5f}")
print(f"Without {OMITTED}: {ommited_recall:.5f}")
display(differences)

Total false negatives: 50
L
L = 0    50
L = 1     0
Name: count, dtype: int64
Accuracies:
Prior only: 0.09674
Full sensors: 0.98289
Without L: 0.95459

Precisions:
Prior only: 0.09674
Full sensors: 0.99187
Without L: 0.68140

Recalls:
Prior only: 1.00000
Full sensors: 0.82993
Without L: 0.99660


,n,brier,mean_loss,TN,FP,FN,TP
prior only - full sensors,0.0,0.080780,0.820336,-2743.0,2743.0,-50.0,50.0
prior only - without L,0.0,0.093098,0.856532,-2608.0,2608.0,-1.0,1.0
full sensors - without L,0.0,0.012318,0.036196,135.0,-135.0,49.0,-49.0


## Answers and Analisys

### 1. CPD validation and the meaning of the numerator and denominator

The model check confirms that the fitted Bayesian network is internally consistent: every CPD matches the graph structure, and the probabilities in every CPD column sum to one. The numerical check then focuses on one entry of the motion CPD:

$$P(M=1 \mid O=1)$$

The selected subset contains the values of `M` only for training records where `O=1`:

- `Subset length = 1607`: there are 1,607 training records with at least one occupant.
- `Count of child_state = 939`: in 939 of those records, the motion state is `M=1`.
- `State names for M = [0, 1]`: motion is binary, so there are two possible child states.

With one Dirichlet pseudocount per CPD cell, the posterior-mean estimate is:

$$P(M=1 \mid O=1)$$

$$P(M=1 \mid O=1) = \frac{939 + 1}{1607 + 2} = \frac{940}{1609} \approx 0{,}5842$$

Therefore, the **numerator** is the observed count of the requested child state (`M=1`) plus one pseudocount. The **denominator** is the number of records for the parent configuration (`O=1`) plus one pseudocount for each of the two possible states of `M`, giving `1607 + 2`.

The manually calculated value, `0.584213797389683`, is exactly the same as the value returned by `pgmpy` for the corresponding CPD entry. This confirms that the model learned the expected smoothed conditional probability:


$$P(M=1\mid O=1)\approx 0.5842.$$

### 2. Experimental comparison

The experiment evaluates the same fitted model on the same 3,039 test records under three evidence conditions:

1. **Prior only:** no sensor evidence is used. Every record receives the fitted prior probability of occupancy.
2. **Full sensors:** all five sensor features (`T`, `L`, `S`, `C`, and `M`) are used.
3. **Without L:** the light sensor is omitted, while the other four sensors are retained.

The training data, fitted CPDs, test records, decision threshold, and evaluation metrics are kept fixed. Consequently, the direct comparison between **full sensors** and **without L** isolates the effect of omitting the assigned light sensor. The prior-only condition is a baseline for measuring the value of using sensor evidence, but it is not an outage condition.

The decision rule is to classify a record as occupied when:

$$p(O=1\mid\text{evidence})\geq \frac{1}{6}.$$

The cost-sensitive mean loss is:

$$\text{mean loss}=\frac{FP+5FN}{n},$$

because a false negative is assigned five times the cost of a false positive.

#### Overall results

| Condition | Brier | Mean loss | Accuracy | Precision | Recall | TN | FP | FN | TP |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Prior only | 0.10428 | 0.90326 | 0.09674 | 0.09674 | 1.00000 | 0 | 2745 | 0 | 294 |
| Full sensors | 0.02350 | 0.08292 | 0.98289 | 0.99187 | 0.82993 | 2743 | 2 | 50 | 244 |
| Without L | 0.01118 | 0.04673 | 0.95459 | 0.68140 | 0.99660 | 2608 | 137 | 1 | 293 |

The prior-only model predicts all records as occupied because the fitted prior is above the threshold. This explains its confusion matrix: it has 294 true positives and no false negatives, but it also labels all 2,745 truly empty records as occupied, producing 2,745 false positives. Its recall is therefore perfect, but its precision and accuracy are both only `0.09674`.

Using the full sensor evidence greatly improves the probabilistic predictions. The Brier score decreases from `0.10428` to `0.02350`, an improvement of `0.08078`, and the mean loss decreases from `0.90326` to `0.08292`, an improvement of `0.82034`. Classification accuracy rises to `0.98289`, with 2,743 true negatives and only 2 false positives. This produces a very high precision of `0.99187`. However, 50 occupied records fall below the threshold, so recall is `0.82993` rather than perfect.

#### Effect of omitting the light sensor

The most important comparison is:

$$\text{full sensors} - \text{without L}.$$

Both conditions use the same fitted model and the same test records; only the light evidence is removed. Without `L`, the Brier score decreases from `0.02350` to `0.01118`, and the mean loss decreases from `0.08292` to `0.04673`. In this dataset, the model is therefore better calibrated in the Brier-score sense and obtains a lower cost-sensitive loss when `L` is omitted.

The confusion matrix shows why the result is not simply “full sensors are better”:

- Omitting `L` changes `TN` from 2,743 to 2,608, so it creates 135 additional false positives (`FP` changes from 2 to 137).
- At the same time, it changes `FN` from 50 to 1, reducing false negatives by 49.
- `TP` increases from 244 to 293, so 49 more occupied records are detected.
- Because false negatives cost five times more than false positives, reducing 49 false negatives is more valuable under the stated loss than creating 135 false positives:

$$\frac{2+5(50)}{3039}=0.08292,
\qquad
\frac{137+5(1)}{3039}=0.04673.$$

This explains the lower mean loss without `L`, despite its lower accuracy and much lower precision. The omitted-light model is more willing to classify records as occupied: this increases recall to `0.99660`, but also produces more false alarms, reducing precision to `0.68140`.

The false-negative analysis supports this interpretation. All 50 full-sensor false negatives occur when `L=0`, and none occurs when `L=1`. Thus, for these 50 occupied records, the observed light state is associated with evidence that contributes to a posterior below the `1/6` decision threshold. This does not mean that `L=0` causes the errors or that light is the only relevant feature; the other sensor values and the learned CPDs also contribute to every posterior.

#### Prior baseline versus sensor conditions

Relative to the prior-only baseline, the full-sensor condition changes the metrics by:

- Brier: `-0.08078`;
- mean loss: `-0.82034`;
- TN: `+2743`;
- FP: `-2743`;
- FN: `+50`;
- TP: `-50`.

The omitted-light condition changes them by:

- Brier: `-0.09310`;
- mean loss: `-0.85653`;
- TN: `+2608`;
- FP: `-2608`;
- FN: `+1`;
- TP: `-1`.

These comparisons show that both sensor-based models are much more useful than the prior-only baseline. They replace the prior model's strategy of declaring every record occupied with evidence-based decisions that correctly reject most empty records.

#### Interpretation and limitation

Under the assignment's specified cost structure, omitting `L` gives the lowest mean loss on this fixed test set because the reduction in false negatives outweighs the increase in false positives. However, this is not evidence that removing the light sensor is generally preferable. The result is specific to this dataset, this fixed train/test split, this model structure, the discretisation, the smoothing choice, the threshold of `1/6`, and the chosen costs.

The experiment does not establish causal effects, general performance on other buildings or time periods, or that `L` is intrinsically harmful. It only measures how this already-fitted model behaves on these test records when the light evidence is available versus unavailable. A broader conclusion would require additional time-based validation or repeated evaluation under unchanged modelling decisions.

In [37]:
# A reproducible incident per group, assigned without inspecting model scores.
group_index = int(GROUP[1:]) - 1
incident_position = min(len(test)-1, int((group_index + 0.5) * len(test) / 19))
incident = test.iloc[incident_position]
evidence_full = make_evidence(incident[FEATURES], None)
evidence_outage = make_evidence(incident[FEATURES], OMITTED)
p_full = occupancy_probability(evidence_full)
p_outage = occupancy_probability(evidence_outage)
display(pd.DataFrame({
    "condition": ["full sensors", f"without {OMITTED}"],
    "probability": [p_full, p_outage],
    "decision_occupied": [p_full >= THRESHOLD, p_outage >= THRESHOLD],
}))
print("Incident record:", int(incident.record_id), "timestamp:", incident.timestamp)
print("Evidence:", evidence_full, "versus", evidence_outage)

,condition,probability,decision_occupied
0,full sensors,5.066230e-09,False
1,without L,3.507315e-06,False


Incident record: 8929 timestamp: 2018-01-10T22:41:20
Evidence: {'T': 0, 'L': 0, 'S': 1, 'C': 0, 'M': 0} versus {'T': 0, 'S': 1, 'C': 0, 'M': 0}


### Three-slide presentation (Optional)
1. **Model and assumption:** graph, variable meanings and one modelling assumption.
2. **Protocol and evidence:** the fixed split, readable prior/full/outage results table,
   and one correctly interpreted numerical result.
3. **Engineering conclusion:** the incident or outage consequence, an important
   limitation and one further validation.

### Data attribution
Adarsh Pal Singh and Sachin Chaudhari, **Room Occupancy Estimation**, UCI Machine
Learning Repository, DOI **10.24432/C5P605**, CC BY 4.0. The binary target, same-row
sensor aggregates, training-only discretisation and fixed split are course adaptations.